In [ ]:
%load_ext autoreload
%autoreload 2

import os
import torch
import torch.nn as nn
import torch.optim as optim
import math
import shutil
from tqdm.notebook import tqdm
import numpy as np
import skimage
from skimage.metrics import structural_similarity as ssim

def compute_rmse(pred, gt):
    return torch.sqrt(torch.mean((pred - gt) ** 2))

import torch.nn.functional as F
from hann_merge_vdsr import HannStreamMerger

# Import your custom modules directly from your folder
from dataloader_dem import create_dataloaders

# from model_vdsr import BaselineVDSR, GeomorphologicalLoss
from model_vdsr import BaselineVDSR, GeomorphologicalLoss
# ==========================================
# GLOBAL HYPERPARAMETERS
# ==========================================
ELEVATION_SCALE = 100.0  # Shrink terrain by 100x for the network


# ==========================================
# 1. CHECKPOINTING LOGIC (ROBUST SAVE & RESUME)
# ==========================================

CHECKPOINT_DIR = "vdsr_checkpoints"


def move_optimizer_to_device(optimizer, device):
    """
    Important for resume-on-GPU:
    optimizer state tensors are often loaded on CPU and must be moved.
    """
    for state in optimizer.state.values():
        for k, v in state.items():
            if torch.is_tensor(v):
                state[k] = v.to(device)


def resolve_resume_path(resume_path=None, save_dir=CHECKPOINT_DIR):
    """
    resume_path options:
      - None / False : start fresh
      - "auto" or "latest" : resume from checkpoint_latest.pth if present
      - explicit path : use that exact checkpoint
    """
    latest_path = os.path.join(save_dir, "checkpoint_latest.pth")

    if resume_path in [None, False, ""]:
        return None

    if isinstance(resume_path, str) and resume_path.lower() in ["auto", "latest"]:
        if os.path.isfile(latest_path):
            return latest_path
        print(f"=> No latest checkpoint found at: {latest_path}")
        return None

    if os.path.isfile(resume_path):
        return resume_path

    raise FileNotFoundError(f"Resume checkpoint not found: {resume_path}")


def save_checkpoint(state, is_best, save_dir=CHECKPOINT_DIR):
    """
    Saves:
      - checkpoint_latest.pth
      - checkpoint_epoch_XXXX.pth
      - checkpoint_best_rmse.pth (if improved)
    """
    os.makedirs(save_dir, exist_ok=True)

    epoch = int(state["epoch"])
    latest_path = os.path.join(save_dir, "checkpoint_latest.pth")
    epoch_path = os.path.join(save_dir, f"checkpoint_epoch_{epoch+1:04d}.pth")
    best_path = os.path.join(save_dir, "checkpoint_best_rmse.pth")

    # Save latest
    torch.save(state, latest_path)

    # Save archival per-epoch checkpoint
    shutil.copyfile(latest_path, epoch_path)

    # Save best checkpoint if improved
    if is_best:
        shutil.copyfile(latest_path, best_path)


def load_checkpoint(checkpoint_path, model, optimizer=None, scheduler=None, device="cpu"):
    print(f"=> Resuming from: {checkpoint_path}")

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    if "model_state_dict" not in checkpoint:
        raise KeyError("Checkpoint missing 'model_state_dict'.")

    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)

    if optimizer is not None:
        if "optimizer_state_dict" not in checkpoint:
            raise KeyError("Checkpoint missing 'optimizer_state_dict'.")
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        move_optimizer_to_device(optimizer, device)

    if scheduler is not None:
        if "scheduler_state_dict" not in checkpoint:
            raise KeyError("Checkpoint missing 'scheduler_state_dict'.")
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    start_epoch = int(checkpoint.get("epoch", -1)) + 1
    best_rmse = float(checkpoint.get("best_loss", float("inf")))

    print(f"=> Loaded epoch: {start_epoch}")
    print(f"=> Best RMSE so far: {best_rmse:.4f} m")

    return start_epoch, best_rmse


# ==========================================
# 2. VALIDATION LOGIC (Mathematically Pure)
# ==========================================
def calculate_psnr(mse, max_val):
    if mse == 0: return float('inf')
    return 20 * math.log10(max_val) - 10 * math.log10(mse)

# ==========================================
# VALIDATION TILE / HANN MERGE HELPERS
# ==========================================

# ==========================================
# VALIDATION TILE / HANN MERGE HELPERS
# ==========================================

VAL_PATCH_SIZE = 256
VAL_OVERLAP = 192
VAL_STRIDE = VAL_PATCH_SIZE - VAL_OVERLAP   # 64
VAL_TILE_BATCH_SIZE = 32
VAL_MERGE_DEVICE = "cpu"                    # safest for VRAM


def _make_starts(length, patch_size, stride):
    if length <= patch_size:
        return [0]

    starts = list(range(0, length - patch_size + 1, stride))
    last = length - patch_size

    if starts[-1] != last:
        starts.append(last)

    return starts


def _extra_pad_to_fit_grid(base_len, patch_size, stride):
    target_len = max(base_len, patch_size)
    remainder = (target_len - patch_size) % stride
    extra = (stride - remainder) % stride
    return (target_len + extra) - base_len


def _pad_for_hann_validation(x_abs, patch_size=256, overlap=192):
    """
    Pads the absolute DEM so Hann overlap-add reconstructs borders safely.
    """
    stride = patch_size - overlap
    border_pad = overlap // 2   # 96

    h, w = x_abs.shape[-2:]
    base_h = h + 2 * border_pad
    base_w = w + 2 * border_pad

    extra_bottom = _extra_pad_to_fit_grid(base_h, patch_size, stride)
    extra_right  = _extra_pad_to_fit_grid(base_w, patch_size, stride)

    padded = F.pad(
        x_abs,
        pad=(border_pad, border_pad + extra_right, border_pad, border_pad + extra_bottom),
        mode="replicate",
    )

    return padded, border_pad


@torch.inference_mode()
def run_hann_tiled_inference(
    model,
    lr_dem_centered,
    global_mean,
    device,
    patch_size=256,
    overlap=192,
    tile_batch_size=8,
    merger_device="cpu",
    show_progress=False,
    file_idx=None,
    num_files=None,
    tqdm_position=2,
):
    """
    Patch-wise validation inference + Hann overlap-add merge.

    Args:
        lr_dem_centered: [1, 1, H, W]
        global_mean:     [1]
    Returns:
        pred_real: [1, 1, H, W] on CPU
    """
    assert lr_dem_centered.shape[0] == 1, "Validation loader must use batch_size=1."

    stride = patch_size - overlap

    # Restore absolute DEM on CPU first
    lr_abs = lr_dem_centered.cpu() + global_mean.view(1, 1, 1, 1).cpu()
    original_shape = tuple(lr_abs.shape[-2:])

    # Pad so borders are reconstructed safely with Hann
    padded_lr_abs, border_pad = _pad_for_hann_validation(
        lr_abs,
        patch_size=patch_size,
        overlap=overlap,
    )

    H_pad, W_pad = padded_lr_abs.shape[-2:]

    merger = HannStreamMerger(
        canvas_shape=(H_pad, W_pad),
        patch_size=patch_size,
        device=merger_device,
        pad=border_pad,
        original_shape=original_shape,
    )

    y_starts = _make_starts(H_pad, patch_size, stride)
    x_starts = _make_starts(W_pad, patch_size, stride)
    total_tiles = len(y_starts) * len(x_starts)

    patch_bar = None
    if show_progress:
        if file_idx is not None and num_files is not None:
            patch_desc = f"Val patches [{file_idx+1}/{num_files}]"
        else:
            patch_desc = "Val patches"

        patch_bar = tqdm(
            total=total_tiles,
            desc=patch_desc,
            position=tqdm_position,
            leave=False,
            dynamic_ncols=True,
        )

    patch_batch = []
    mean_batch = []
    coord_batch = []

    def flush_batch():
        if len(patch_batch) == 0:
            return

        inp = torch.cat(patch_batch, dim=0).to(device, non_blocking=True)
        inp = inp / ELEVATION_SCALE

        preds_centered = model(inp) * ELEVATION_SCALE

        merger.add_batch(
            preds_centered=preds_centered.detach(),
            patch_means=torch.stack(mean_batch),
            coords=torch.tensor(coord_batch, dtype=torch.long),
        )

        patch_batch.clear()
        mean_batch.clear()
        coord_batch.clear()

    for y in y_starts:
        for x in x_starts:
            patch_abs = padded_lr_abs[:, :, y:y + patch_size, x:x + patch_size]

            # Patch-local centering to match training behavior
            local_mean = patch_abs.mean(dim=(1, 2, 3))   # [1]
            patch_centered = patch_abs - local_mean.view(1, 1, 1, 1)

            patch_batch.append(patch_centered)
            mean_batch.append(local_mean.squeeze(0))
            coord_batch.append([y, x])

            if len(patch_batch) >= tile_batch_size:
                flush_batch()

            if patch_bar is not None:
                patch_bar.update(1)

    flush_batch()

    if patch_bar is not None:
        patch_bar.close()

    pred_real = merger.get_final_dem(as_tensor=True).unsqueeze(0).unsqueeze(0).cpu()
    return pred_real

def evaluate_vdsr(
    model,
    val_loader,
    device,
    shave_size=3,
    patch_size=VAL_PATCH_SIZE,
    overlap=VAL_OVERLAP,
    tile_batch_size=VAL_TILE_BATCH_SIZE,
    merger_device=VAL_MERGE_DEVICE,
    show_progress=True,
    tqdm_position=1,
):
    model.eval()

    total_sq_error = 0.0
    total_abs_error = 0.0
    total_slope_rmse = 0.0
    total_ssim = 0.0
    total_psnr = 0.0

    total_pixels = 0
    total_patches = 0
    total_batches = 0

    sobel_x = (
        torch.tensor(
            [[-1., 0., 1.],
             [-2., 0., 2.],
             [-1., 0., 1.]],
            dtype=torch.float32,
        ).view(1, 1, 3, 3) / 8.0
    )

    sobel_y = (
        torch.tensor(
            [[-1., -2., -1.],
             [ 0.,  0.,  0.],
             [ 1.,  2.,  1.]],
            dtype=torch.float32,
        ).view(1, 1, 3, 3) / 8.0
    )

    val_iter = enumerate(val_loader)
    if show_progress:
        val_iter = tqdm(
            val_iter,
            total=len(val_loader),
            desc="Validation files",
            position=tqdm_position,
            leave=False,
            dynamic_ncols=True,
        )

    with torch.no_grad():
        for file_idx, (lr_dem, hr_gt, patch_mean) in val_iter:
            # --------------------------------------------
            # 1) Tiled inference + Hann merge
            # --------------------------------------------
            pred_real = run_hann_tiled_inference(
                model=model,
                lr_dem_centered=lr_dem,
                global_mean=patch_mean,
                device=device,
                patch_size=patch_size,
                overlap=overlap,
                tile_batch_size=tile_batch_size,
                merger_device=merger_device,
                show_progress=show_progress,
                file_idx=file_idx,
                num_files=len(val_loader),
                tqdm_position=tqdm_position + 1,
            )

            # Restore GT to absolute elevations on CPU
            hr_gt_real = hr_gt.cpu() + patch_mean.view(-1, 1, 1, 1).cpu()

            if shave_size > 0:
                pred_real = pred_real[:, :, shave_size:-shave_size, shave_size:-shave_size]
                hr_gt_real = hr_gt_real[:, :, shave_size:-shave_size, shave_size:-shave_size]

            # --------------------------------------------
            # 2) Slope RMSE
            # --------------------------------------------
            p_dx = F.conv2d(pred_real, sobel_x, padding=1)
            p_dy = F.conv2d(pred_real, sobel_y, padding=1)

            gt_dx = F.conv2d(hr_gt_real, sobel_x, padding=1)
            gt_dy = F.conv2d(hr_gt_real, sobel_y, padding=1)

            p_slope = torch.sqrt(p_dx ** 2 + p_dy ** 2)
            gt_slope = torch.sqrt(gt_dx ** 2 + gt_dy ** 2)

            total_slope_rmse += compute_rmse(p_slope, gt_slope).item()
            total_batches += 1

            # --------------------------------------------
            # 3) Global RMSE / MAE
            # --------------------------------------------
            p_flat = pred_real.flatten()
            gt_flat = hr_gt_real.flatten()

            total_pixels += p_flat.numel()
            total_sq_error += torch.sum((p_flat - gt_flat) ** 2).item()
            total_abs_error += torch.sum(torch.abs(p_flat - gt_flat)).item()

            # --------------------------------------------
            # 4) PSNR / SSIM
            # --------------------------------------------
            mse_b = torch.mean((p_flat - gt_flat) ** 2).item()
            max_b = max((torch.max(hr_gt_real) - torch.min(hr_gt_real)).item(), 1.0)

            total_psnr += calculate_psnr(mse_b, max_b)

            p_np = pred_real.squeeze().numpy()
            gt_np = hr_gt_real.squeeze().numpy()
            ssim_val = ssim(p_np, gt_np, data_range=max_b)
            total_ssim += ssim_val

            total_patches += 1

            if show_progress and hasattr(val_iter, "set_postfix"):
                current_rmse = math.sqrt(total_sq_error / max(total_pixels, 1))
                val_iter.set_postfix({
                    "done": f"{file_idx+1}/{len(val_loader)}",
                    "RMSE": f"{current_rmse:.2f}m"
                })

            if device.type == "cuda":
                torch.cuda.empty_cache()

    global_rmse = math.sqrt(total_sq_error / total_pixels)
    global_mae = total_abs_error / total_pixels
    avg_psnr = total_psnr / total_patches
    avg_ssim = total_ssim / total_patches
    avg_slope_rmse = total_slope_rmse / max(total_batches, 1)

    return global_rmse, global_mae, avg_psnr, avg_ssim, avg_slope_rmse


# ==========================================
# 3. TRAINING LOOP (Handbrake Released)
# ==========================================
def train_vdsr(
    model,
    train_loader,
    val_loader,
    num_epochs=100,
    resume_path=None,
    save_dir=CHECKPOINT_DIR,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Engaging Hardware: {device}")

    model = model.to(device)

    criterion = GeomorphologicalLoss(alpha=1.0, beta=5.0, gamma=15.0).to(device)

    optimizer = optim.Adam(model.parameters(), lr=2e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs,
        eta_min=1e-6
    )

    start_epoch = 0
    best_rmse = float("inf")

    # -----------------------------
    # Resume logic
    # -----------------------------
    resolved_resume = resolve_resume_path(resume_path, save_dir=save_dir)
    if resolved_resume is not None:
        start_epoch, best_rmse = load_checkpoint(
            resolved_resume,
            model,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
        )
    else:
        print("=> Starting fresh training run.")

    if start_epoch >= num_epochs:
        print(f"=> Checkpoint already at epoch {start_epoch}, which is >= num_epochs={num_epochs}. Nothing to train.")
        return

    epoch_bar = tqdm(
        range(start_epoch, num_epochs),
        total=(num_epochs - start_epoch),
        desc="Epochs",
        position=0,
        leave=True,
        dynamic_ncols=True,
    )

    for epoch in epoch_bar:
        model.train()

        train_loss_accumulator = 0.0
        train_base_acc = 0.0
        train_slope_acc = 0.0
        train_curve_acc = 0.0

        train_bar = tqdm(
            train_loader,
            desc=f"Train batches [{epoch+1}/{num_epochs}]",
            position=1,
            leave=False,
            dynamic_ncols=True,
        )

        for lr_dem, hr_gt, patch_mean in train_bar:
            lr_dem = lr_dem.to(device, non_blocking=True)
            hr_gt = hr_gt.to(device, non_blocking=True)

            optimizer.zero_grad()

            preds_scaled = model(lr_dem / ELEVATION_SCALE)
            gt_scaled = hr_gt / ELEVATION_SCALE

            loss, b_loss, s_loss, c_loss = criterion(preds_scaled, gt_scaled)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()

            with torch.no_grad():
                pure_mae_meters = torch.mean(torch.abs(preds_scaled - gt_scaled)).item() * ELEVATION_SCALE

            train_loss_accumulator += pure_mae_meters
            train_base_acc += b_loss.item()
            train_slope_acc += s_loss.item()
            train_curve_acc += c_loss.item()

            train_bar.set_postfix({
                "Pure_MAE": f"{pure_mae_meters:.2f}m",
                "TopoLoss": f"{loss.item():.4f}"
            })

        avg_train_mae = train_loss_accumulator / len(train_loader)
        avg_base = train_base_acc / len(train_loader)
        avg_slope = train_slope_acc / len(train_loader)
        avg_curve = train_curve_acc / len(train_loader)

        # --------------------------------
        # Validation
        # --------------------------------
        val_rmse, val_mae, val_psnr, val_ssim, val_slope_rmse = evaluate_vdsr(
            model,
            val_loader,
            device,
            shave_size=3,
            patch_size=256,
            overlap=192,
            tile_batch_size=8,
            merger_device="cpu",
            show_progress=True,
            tqdm_position=1,
        )

        scheduler.step()

        is_best = val_rmse < best_rmse
        if is_best:
            best_rmse = val_rmse

        # --------------------------------
        # Save checkpoint every epoch
        # --------------------------------
        checkpoint_state = {
            "epoch": epoch,
            "num_epochs": num_epochs,
            "best_loss": best_rmse,
            "last_val_rmse": val_rmse,
            "last_val_mae": val_mae,
            "last_val_psnr": val_psnr,
            "last_val_ssim": val_ssim,
            "last_val_slope_rmse": val_slope_rmse,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
        }

        save_checkpoint(
            checkpoint_state,
            is_best=is_best,
            save_dir=save_dir
        )

        epoch_bar.set_postfix({
            "LR": f"{scheduler.get_last_lr()[0]:.2e}",
            "Val_RMSE": f"{val_rmse:.3f}m",
            "Best_RMSE": f"{best_rmse:.3f}m"
        })

        tqdm.write(f"Epoch {epoch+1:03d} | LR: {scheduler.get_last_lr()[0]:.2e}")
        tqdm.write(
            f"  ├─ Train Losses -> Base: {avg_base:.4f} | "
            f"Slope: {avg_slope:.4f} | Curve: {avg_curve:.4f} | "
            f"Pure MAE: {avg_train_mae:.4f}m"
        )
        tqdm.write(
            f"  └─ Val Metrics  -> RMSE: {val_rmse:.4f}m | "
            f"MAE: {val_mae:.4f}m | PSNR: {val_psnr:.2f}dB | "
            f"SSIM: {val_ssim:.4f} | Slope RMSE: {val_slope_rmse:.4f}"
        )

In [ ]:
# 1. Clear VRAM 
torch.cuda.empty_cache()

# 2. Map your specific dataset folders
train_dirs = [
    r"D:\Vaibhav\vdsr-atl08\Dataset\Kl\tensors\train",
    r"D:\Vaibhav\vdsr-atl08\Dataset\SG\tensors\train"
]
val_dirs = [
    r"D:\Vaibhav\vdsr-atl08\Dataset\Kl\tensors\validation_contiguous",
    r"D:\Vaibhav\vdsr-atl08\Dataset\SG\tensors\validation_contiguous"
]

# 3. Build the loaders
train_loader, val_loader = create_dataloaders(train_dirs, val_dirs)

# 4. Instantiate the model
# model = BaselineVDSR(num_layers=sr20, num_features=64)
model = BaselineVDSR(num_layers=20, num_features=64)# 5. Start Training!
train_vdsr(
    model,
    train_loader,
    val_loader,
    num_epochs=500,
    resume_path="vdsr_checkpoints/checkpoint_best_rmse.pth"
)